# ER-CyRIS Siklus 2 — Koreksi evaluasi kandidat M1 sampai M4 pada data pengembangan

**Versi v2, terpisah dari v1.** Dua perbaikan: gerbang mutu tidak dapat menaikkan `abstain` menjadi `warn`; pembanding detektor menggunakan alert aktual pada setiap kondisi. Kohor alert bersih tetap dilaporkan secara terpisah sebagai diagnostik berpasangan. Semua ambang dan definisi `q_e` tetap seperti v1; v2 **tidak** menyetel ambang untuk mempercantik hasil development. File2 tetap tertutup. Arsip v1 tidak diubah.

`q_e` di sini hanya kelengkapan 43 fitur, `m_d` hanya margin terhadap ambang detektor, dan `r_x` hanya kestabilan SHAP terhadap gangguan numerik kecil. Baik kualitas konteks organisasi, risiko, maupun respons insiden belum diuji. Target utama v2 adalah mengoreksi validitas pembandingan; penurunan cakupan karena `q_e` pada v1 tidak boleh disembunyikan.


In [ ]:
import csv, hashlib, json, platform
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import shap
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/ercyris_siklus2')
except ImportError:
    BASE=Path('.')
P1=BASE/'UNSW-NB15_1.csv'
LOCK_DIR=BASE/'unsw_locked_protocol_v1'
LOCK=LOCK_DIR/'unsw_lock_summary.json'
MANIFEST=LOCK_DIR/'unsw_locked_split_manifest.csv'
OUT=BASE/'unsw_framework_dev_v2'
OUTFILE=OUT/'unsw_framework_development_v2.json'
SEED=42; MISSING_RATE=.10; PROBE_SIGMA=.01; SHAP_MAX=300; TOPK=10
TAU_Q=.95; TAU_Q_SEVERE=.70; MIN_GATE_COVERAGE=.50
for path in (P1,LOCK,MANIFEST):
    if not path.is_file(): raise FileNotFoundError(f'Dibutuhkan: {path}')
if OUTFILE.exists():raise FileExistsError('Keluaran v2 sudah ada; STOP agar tidak menimpa.')
print('Output:',OUTFILE,'| File2 tidak akan dibuka.')

## 1. Verifikasi lock dan muat hanya baris file1

Identitas dua sumber tetap tercatat dalam lock, tetapi notebook ini membuka hanya file1. Kesamaan SHA sumber dan manifest, partisi, nomor baris, dan label diperiksa sebelum pelatihan. Bila ada ketidakcocokan, eksekusi berhenti.

In [ ]:
COLS=['srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes',
      'sttl','dttl','sloss','dloss','service','sload','dload','spkts','dpkts',
      'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth',
      'res_bdy_len','sjit','djit','stime','ltime','sintpkt','dintpkt',
      'tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl',
      'ct_flw_http_mthd','is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst',
      'ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
      'ct_dst_src_ltm','attack_cat','label']
EXCLUDE={'label','attack_cat','srcip','dstip','stime','ltime'}
FEATURES=[c for c in COLS if c not in EXCLUDE]
CATS=['proto','state','service'];NUMS=[c for c in FEATURES if c not in CATS]
assert len(FEATURES)==43

def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for piece in iter(lambda:f.read(4*1024*1024),b''):h.update(piece)
    return h.hexdigest()

lock=json.loads(LOCK.read_text(encoding='utf-8'))
if lock['status']!='FROZEN_SPLIT_NO_MODEL_NO_HOLDOUT_EVALUATION':
    raise ValueError('Status lock tidak sesuai; STOP.')
if sha256(P1)!=lock['sources']['file1']['sha256'] or sha256(MANIFEST)!=lock['manifest']['sha256']:
    raise ValueError('Berkas sumber atau manifest berubah sejak lock; STOP.')
manifest=pd.read_csv(MANIFEST,dtype={'feature_group_hash64':str})
if len(manifest)!=lock['manifest']['row_count']:
    raise ValueError('Jumlah baris manifest berubah; STOP.')
local=manifest.loc[manifest.source_file==P1.name].copy()
if set(local.partition)!={'train','calibration','development'} or len(local)!=30000:
    raise ValueError('Partisi file1 salah; STOP.')
if manifest.loc[manifest.partition=='holdout','source_file'].nunique()!=1:
    raise ValueError('Holdout bukan tepat satu file; STOP.')
if local.source_row_zero_based.duplicated().any():
    raise ValueError('Nomor baris file1 duplikat lintas partisi; STOP.')
with P1.open('r',encoding='utf-8-sig',errors='replace',newline='') as f:
    r=next(csv.reader(f))
if len(r)!=49 or r[-1].strip() not in ('0','1'):
    raise ValueError('Skema sumber tidak cocok; STOP.')
needed=set(local.source_row_zero_based.astype(int)); chunks=[]; offset=0
for chunk in pd.read_csv(P1,header=None,names=COLS,dtype=str,na_filter=False,chunksize=100000):
    positions=np.arange(offset,offset+len(chunk),dtype=np.int32)
    yes=np.isin(positions,list(needed))
    if yes.any():
        z=chunk.loc[yes].copy();z.index=positions[yes];chunks.append(z)
    offset+=len(chunk)
if offset!=lock['sources']['file1']['n']:raise ValueError('Jumlah baris file1 berbeda.')
raw=pd.concat(chunks).sort_index()
if len(raw)!=len(local):raise ValueError('Baris dalam manifest tidak seluruhnya ditemukan.')
labels=pd.to_numeric(raw['label'],errors='coerce')
if labels.isna().any() or not labels.isin([0,1]).all():raise ValueError('Label file1 tidak biner.')
lookup=local.set_index('source_row_zero_based').sort_index()
if not np.array_equal(labels.to_numpy(dtype=int),lookup['label'].to_numpy(dtype=int)):
    raise ValueError('Label manifest tidak sama dengan label sumber.')
for k in ('train','calibration','development'):
    print(k,lock['partitions'][k]['n'],'positif',lock['partitions'][k]['positives'])
print('Manifest tervalidasi. File2 belum dibaca.')

## 2. M1 kelengkapan dan transformasi M2 train only

Imputasi median, skala, dan penyandian kategori dipelajari dari `train` saja. Metadata provenance meliputi SHA sumber, nomor baris, dan rentang waktu; konteks aset eksternal tidak tersedia. Gangguan missingness yang disuntikkan ke calibration dan development didefinisikan terpisah; ia menguji reaksi aturan atas gangguan sintetis, **bukan** frekuensi gangguan nyata di organisasi.

In [ ]:
def prepare(frame):
    x=frame[FEATURES].copy()
    for c in NUMS:x[c]=pd.to_numeric(x[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    for c in CATS:x[c]=x[c].astype(str).replace('', '__MISSING__')
    return x

def evidence_completeness(x):
    present=x[FEATURES].notna().copy()
    for c in CATS:present[c]&=x[c].astype(str).ne('__MISSING__')
    return present.mean(axis=1).to_numpy(dtype=float)

def synthetic_missing(frame, seed, frac=MISSING_RATE):
    v=frame.copy();rng=np.random.default_rng(seed)
    mask=rng.random((len(v),len(NUMS)))<frac
    for k,c in enumerate(NUMS):v.loc[mask[:,k],c]=np.nan
    return v

parts={name:lookup.loc[lookup.partition==name].index.to_numpy(dtype=np.int32)
       for name in ('train','calibration','development')}
xraw={name:prepare(raw.loc[ix]) for name,ix in parts.items()}
y={name:labels.loc[ix].to_numpy(dtype=np.int8) for name,ix in parts.items()}
xraw['cal_missing']=synthetic_missing(xraw['calibration'],SEED+1)
xraw['dev_missing']=synthetic_missing(xraw['development'],SEED+2)
y['cal_missing']=y['calibration'];y['dev_missing']=y['development']
q={name:evidence_completeness(xraw[name]) for name in xraw}
try:
    enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False,dtype=np.float32)
except TypeError:
    enc=OneHotEncoder(handle_unknown='ignore',sparse=False,dtype=np.float32)
prep=ColumnTransformer([
    ('numeric',Pipeline([('median',SimpleImputer(strategy='median')),
                         ('scale',MinMaxScaler())]),NUMS),
    ('categorical',enc,CATS)
],remainder='drop',sparse_threshold=0)
X={'train':np.asarray(prep.fit_transform(xraw['train']),dtype=np.float32)}
for name in xraw:
    if name!='train':X[name]=np.asarray(prep.transform(xraw[name]),dtype=np.float32)
if any(not np.isfinite(v).all() for v in X.values()):raise ValueError('Vektor fitur nonhingga.')
if len({v.shape[1] for v in X.values()})!=1:raise ValueError('Dimensi antarpartisi berbeda.')
print('Fitur hasil train-only transform:',X['train'].shape[1])
print('q_e completeness clean calibration:',float(q['calibration'].mean()),
      'missingness synthetic:',float(q['cal_missing'].mean()))

## 3. Fungsi M2 M3 dan aturan M4

Ambang detektor dipilih hanya dari calibration menggunakan F1, dengan FAR dan kedekatan ke 0,5 sebagai pemutus seri. `m_d` memakai rumus margin terdokumentasi pada Bab VI. `r_x` memakai rata-rata Top-K Jaccard, kestabilan magnitudo, dan konsistensi tanda. Gerbang tiga aspek mempertahankan keputusan dua aspek ketika `q_e ≥ 0,95`; kelengkapan yang lebih rendah memaksa `warn`, dan `q_e < 0,70` memaksa `abstain`. Dua ambang kelengkapan adalah pilihan **protokol kandidat** yang diumumkan sebelum mengukur development, bukan hasil optimasi validitas sumber. Konflik label dan konteks aset belum disimpulkan dari angka ini.

In [ ]:
def select_detector_threshold(y_true,p):
    candidates=np.linspace(.05,.95,19)
    scored=[]
    for t in candidates:
        pred=(p>=t).astype(int)
        tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
        f=f1_score(y_true,pred,zero_division=0)
        far=float(fp/(fp+tn)) if fp+tn else 1.
        scored.append((float(f),-far,-abs(float(t)-.5),float(t)))
    return max(scored)[3]

def margin_d(probability,detector_threshold):
    p=np.clip(np.asarray(probability,dtype=float),0.,1.)
    return np.clip((p-detector_threshold)/max(1.-detector_threshold,1e-12),0.,1.)

def add_numeric_probe(x,seed,sigma=PROBE_SIGMA):
    v=np.asarray(x,dtype=np.float32).copy();rng=np.random.default_rng(seed)
    v[:,:len(NUMS)]=np.clip(v[:,:len(NUMS)]+rng.normal(0,sigma,(len(v),len(NUMS))),0,1)
    return v

def tree_shap_positive(model,x):
    try:
        v=shap.TreeExplainer(model).shap_values(x,check_additivity=False)
    except TypeError:
        v=shap.TreeExplainer(model).shap_values(x)
    if isinstance(v,list):v=v[1]
    v=np.asarray(v,dtype=float)
    if v.ndim==3:
        if v.shape[-1]!=2:raise ValueError(f'SHAP expected binary class, got {v.shape}')
        v=v[:,:,1]
    if v.shape!=x.shape or not np.isfinite(v).all():
        raise ValueError(f'SHAP gagal atau bentuknya berbeda: {v.shape} vs {x.shape}')
    return v

def explanation_stability(clean,probe,k=TOPK):
    a=np.asarray(clean,dtype=float);b=np.asarray(probe,dtype=float)
    if a.shape!=b.shape or a.ndim!=2:raise ValueError('SHAP dua kondisi harus seukuran.')
    k=min(int(k),a.shape[1]);
    ac=np.abs(a);bc=np.abs(b)
    ta=np.argsort(-ac,axis=1)[:,:k];tb=np.argsort(-bc,axis=1)[:,:k]
    jac=[];sign=[]
    for i,(left,right) in enumerate(zip(ta,tb)):
        u=set(left)|set(right);jac.append(len(set(left)&set(right))/len(u))
        pos=sorted(u);sign.append(float(np.mean(np.sign(a[i,pos])==np.sign(b[i,pos]))))
    den=ac.mean(axis=1)+1e-12
    mag=np.clip(1-np.abs(ac-bc).mean(axis=1)/den,0,1)
    components=np.column_stack([jac,mag,sign]);return np.clip(components,0,1).mean(axis=1)

def gate2(m_d,r_x,tau_d,tau_x):
    a=np.asarray(m_d)>=tau_d;b=np.asarray(r_x)>=tau_x
    return np.where(a&b,'accept',np.where(a|b,'warn','abstain'))

def gate3(q_e,m_d,r_x,tau_d,tau_x,tau_q=TAU_Q,severe=TAU_Q_SEVERE):
    q_e=np.asarray(q_e,dtype=float)
    if not np.isfinite(q_e).all() or np.any((q_e<0)|(q_e>1)):
        raise ValueError('q_e invalid; STOP.')
    out=gate2(m_d,r_x,tau_d,tau_x)
    # Peringatan mutu hanya menurunkan accept. Abstain tidak boleh meningkat menjadi warn.
    out=np.where((q_e<tau_q)&(out=='accept'),'warn',out)
    return np.where(q_e<severe,'abstain',out)

def select_gate(cal_frame,min_coverage=MIN_GATE_COVERAGE):
    if cal_frame.empty or set(cal_frame['split'])!={'calibration'}:
        raise ValueError('Ambang gate hanya boleh di-fit pada calibration alerts.')
    quantiles=np.linspace(0,1,11)
    ds=np.unique(np.quantile(cal_frame.m_d,quantiles))
    xs=np.unique(np.quantile(cal_frame.r_x,quantiles))
    candidates=[]
    for td in ds:
        for tx in xs:
            action=gate2(cal_frame.m_d,cal_frame.r_x,td,tx)
            take=action=='accept';coverage=float(take.mean())
            if coverage<min_coverage-1e-12 or not take.any():continue
            unsafe=float((cal_frame.y_true.to_numpy()[take]==0).mean())
            candidates.append((unsafe,-coverage,float(td+tx),float(td),float(tx)))
    if not candidates:raise ValueError('Tak ada pasangan ambang yang memenuhi coverage calibration.')
    _,_,_,td,tx=min(candidates)
    return {'tau_d':td,'tau_x':tx,'min_accept_coverage':min_coverage,
            'selection_split':'calibration'}

def candidate_metrics(frame,action):
    ytrue=frame.y_true.to_numpy(dtype=int);a=np.asarray(action)=='accept'
    n=int(len(frame));na=int(a.sum());tp=int((a&(ytrue==1)).sum());fp=int((a&(ytrue==0)).sum())
    alltp=int((ytrue==1).sum())
    if not np.isin(action,['accept','warn','abstain','no_alert']).all():
        raise ValueError('Tindakan tidak dikenal; STOP.')
    return {'candidate_alerts':n,'accept':na,'warn':int((action=='warn').sum()),
            'abstain':int((action=='abstain').sum()),
            'no_alert':int((action=='no_alert').sum()),'accept_coverage':na/n if n else None,
            'false_accept_per_candidate':fp/n if n else None,
            'accepted_precision':tp/na if na else None,
            'retained_true_alert_fraction':tp/alltp if alltp else None,
            'TP_accept':tp,'FP_accept':fp,'total_true_candidates':alltp}

## 4. Train detector dan pilih ambang pada calibration

Model XGBoost dan Random Forest diperbandingkan berpasangan pada partisi yang sama. Hasil yang dicetak di sini masih untuk pengembangan. Perubahan dataset, versi model, dan pemilihan ambang menghasilkan protokol baru; angka tidak boleh ditumpangkan ke arsip lama tanpa identitas versi.

In [ ]:
models={
 'XGBoost':XGBClassifier(random_state=SEED,n_jobs=-1,eval_metric='logloss',verbosity=0,tree_method='hist'),
 'RandomForest':RandomForestClassifier(random_state=SEED,n_jobs=-1),
}
trained={};thresholds={};probs={};detector_scores={}
for name,model in models.items():
    model.fit(X['train'],y['train']);trained[name]=model
    pcal=model.predict_proba(X['calibration'])[:,1]
    threshold=select_detector_threshold(y['calibration'],pcal);thresholds[name]=threshold
    probs[name]={'calibration':pcal}
    detector_scores[name]={}
    for split in ('calibration','development','cal_missing','dev_missing'):
        prob=pcal if split=='calibration' else model.predict_proba(X[split])[:,1]
        probs[name][split]=prob
        pred=prob>=threshold
        tn,fp,fn,tp=confusion_matrix(y[split],pred,labels=[0,1]).ravel()
        detector_scores[name][split]={
            'n':int(len(pred)),'threshold':threshold,
            'F1':float(f1_score(y[split],pred,zero_division=0)),
            'AP':float(average_precision_score(y[split],prob)),
            'FAR':float(fp/(tn+fp)) if tn+fp else None,
            'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
        }
    print(name,'detector threshold calibration=',threshold,
          'F1 clean development=',round(detector_scores[name]['development']['F1'],4))

## 5. Pisahkan evaluasi alert aktual dan diagnostik kohor berpasangan

**Evaluasi operasional:** untuk setiap kondisi, kandidat alert dipilih kembali dari probabilitas model pada kondisi itu. Detektor saja, gerbang dua aspek, dan gerbang tambahan kelengkapan menerima kandidat yang sama. Ambang detektor dan dua aspek dipilih pada calibration bersih, lalu tidak diubah pada development. Ambang `q_e` tetap angka praspesifikasi v1; tidak dikalibrasi ulang pada development.

**Diagnostik berpasangan:** baris yang terdeteksi sebagai alert saat data bersih diikuti pada kondisi kehilangan nilai. Baris yang sudah tidak memenuhi ambang detektor dinyatakan `no_alert` untuk ketiga pembanding. Dengan demikian status pada kohor berpasangan tidak boleh ditafsirkan sebagai cakupan alert aktual tanpa melihat kolom `no_alert`.

Bila sampel alert melebihi batas SHAP, angka hanya berlaku pada sampel yang dilaporkan; tidak ada klaim cakupan seluruh alert. File2 tidak dibuka.


In [ ]:
# Uji logika gerbang harus selesai sebelum pemodelan dan evaluasi.
_qa=np.array([1., .9, .69])
_ma=np.array([1., 1., 1.]);_xa=np.array([1., 1., 1.])
assert list(gate3(_qa,_ma,_xa,.5,.5))==['accept','warn','abstain']
_qa=np.array([.9,.69]);_ma=np.array([0.,0.]);_xa=np.array([0.,0.])
assert list(gate3(_qa,_ma,_xa,.5,.5))==['abstain','abstain']
for _m in (0.,1.):
    for _x in (0.,1.):
        _acts=gate3(np.array([1.,.9,.69]),np.full(3,_m),np.full(3,_x),.5,.5)
        assert all({'abstain':0,'warn':1,'accept':2}[_acts[i]] >=
                   {'abstain':0,'warn':1,'accept':2}[_acts[i+1]] for i in (0,1))

def make_evidence(name,model,split,condition,key,candidate,threshold):
    xi=X[key][candidate]
    local=tree_shap_positive(model,xi)
    probe=tree_shap_positive(model,add_numeric_probe(xi,SEED+5))
    return pd.DataFrame({'split':split,'condition':condition,
          'source_row_zero_based':parts[split][candidate],
          'y_true':y[split][candidate].astype(int),
          'q_e':q[key][candidate],
          'm_d':margin_d(probs[name][key][candidate],threshold),
          'r_x':explanation_stability(local,probe)})

def cap_candidates(candidate,name,split,condition):
    total=len(candidate)
    if total>SHAP_MAX:
        seed_material=f'{SEED}|{name}|{split}|{condition}'.encode('utf-8')
        seed=int.from_bytes(hashlib.sha256(seed_material).digest()[:8],'little')
        candidate=np.sort(np.random.default_rng(seed).choice(candidate,SHAP_MAX,replace=False))
    return candidate,{'detector_positive_before_cap':int(total),
        'shap_evaluated_candidates':int(len(candidate)),
        'sampled':bool(total>SHAP_MAX)}

def evaluate(frame,thresholds,detector_alert=None):
    d=frame.m_d.to_numpy();x=frame.r_x.to_numpy();qe=frame.q_e.to_numpy()
    if detector_alert is None: detector_alert=np.ones(len(frame),dtype=bool)
    detector_alert=np.asarray(detector_alert,dtype=bool)
    if len(detector_alert)!=len(frame):raise ValueError('Mask detektor tidak selaras.')
    baseline=np.where(detector_alert,'accept','no_alert')
    action2=np.where(detector_alert,gate2(d,x,thresholds['tau_d'],thresholds['tau_x']),'no_alert')
    action3=np.where(detector_alert,gate3(qe,d,x,thresholds['tau_d'],thresholds['tau_x']),'no_alert')
    assert np.isin(action3,['accept','warn','abstain','no_alert']).all()
    return {'detector_only':candidate_metrics(frame,baseline),
      'two_domain_gate':candidate_metrics(frame,action2),
      'quality_aware_gate_candidate':candidate_metrics(frame,action3),
      'q_e_mean':float(qe.mean()),
      'q_e_below_warn_threshold':int((qe<TAU_Q).sum()),
      'q_e_below_abstain_threshold':int((qe<TAU_Q_SEVERE).sum())}

evidence={};gate_params={};dynamic_gate_results={};paired_cohort_diagnostic={};candidate_selection={}
for name,model in trained.items():
    threshold=thresholds[name]
    evidence[name]={};candidate_selection[name]={'dynamic':{},'paired_clean_origin':{}}
    # Semua kondisi memakai kandidat alert masing-masing, termasuk data dengan missingness.
    for split,condition,key in [('calibration','clean','calibration'),
                                ('calibration','synthetic_missing','cal_missing'),
                                ('development','clean','development'),
                                ('development','synthetic_missing','dev_missing')]:
        candidate=np.flatnonzero(probs[name][key]>=threshold)
        if not len(candidate):raise ValueError(f'Tidak ada alert aktual untuk {name} {key}.')
        candidate,audit=cap_candidates(candidate,name,split,condition)
        candidate_selection[name]['dynamic'][f'{split}/{condition}']=audit
        evidence[name][(split,condition)]=make_evidence(
            name,model,split,condition,key,candidate,threshold)
    frozen=select_gate(evidence[name][('calibration','clean')]);gate_params[name]=frozen
    dynamic_gate_results[name]={}
    for (split,condition),frame in evidence[name].items():
        dynamic_gate_results[name][f'{split}/{condition}']=evaluate(frame,frozen)
    paired_cohort_diagnostic[name]={}
    for split,missing in (('calibration','cal_missing'),('development','dev_missing')):
        candidate=np.flatnonzero(probs[name][split]>=threshold)
        candidate,audit=cap_candidates(candidate,name,split,'paired_clean_origin')
        candidate_selection[name]['paired_clean_origin'][split]=audit
        # Frame bersih identik dengan evaluasi dinamis bila sampel tidak dibatasi.
        stressed=make_evidence(name,model,split,'synthetic_missing',missing,candidate,threshold)
        current_alert=probs[name][missing][candidate]>=threshold
        paired_cohort_diagnostic[name][f'{split}/synthetic_missing']={
            'cohort_origin':'clean_detector_positive',
            'count_lost_detector_alert':int((~current_alert).sum()),
            'evaluation':evaluate(stressed,frozen,current_alert)}
    print(name,'frozen calibration thresholds:',frozen)
    for key in ('development/clean','development/synthetic_missing'):
        result=dynamic_gate_results[name][key]
        print(' ',key,'actual detector alerts:',result['detector_only']['accept'],
              'two-domain true accepts:',result['two_domain_gate']['TP_accept'],
              'three-aspect true accepts:',result['quality_aware_gate_candidate']['TP_accept'])
print('Uji monotonic gerbang lolos. File2 belum dibaca.')


## 6. Simpan hasil koreksi v2 secara terpisah

`dynamic_gate_results` adalah pembandingan utama alert aktual. `paired_clean_alert_cohort_diagnostic` hanya memeriksa baris yang semula terdeteksi pada kondisi bersih, dengan `no_alert` dihitung eksplisit pada kondisi kehilangan nilai. Nol salah terima pada sampel tidak membuktikan nol salah terima pada populasi. Status penelitian tetap pengembangan dan tidak termasuk uji file2.


In [ ]:
result={
 'status':'DEVELOPMENT_ONLY_CORRECTED_V2_FILE2_UNTOUCHED',
 'source_version':'v1 archived; v2 monotone gate and dynamic candidate correction',
 'lock_protocol_id':lock['protocol_id'],'lock_manifest_sha256':lock['manifest']['sha256'],
 'source_file1_sha256':lock['sources']['file1']['sha256'],
 'config':{'seed':SEED,'models':list(trained),'missing_rate_synthetic':MISSING_RATE,
           'probe_sigma_numeric_transformed':PROBE_SIGMA,'shap_candidate_cap':SHAP_MAX,
           'top_k':TOPK,'tau_q_predefined':TAU_Q,'tau_q_severe_predefined':TAU_Q_SEVERE,
           'q_e_definition':'observed nonmissing features / 43; completeness only',
           'm_d_definition':'clip((p-theta_d)/(1-theta_d),0,1); margin only',
           'r_x_definition':'mean of local SHAP TopK Jaccard, magnitude and sign stability',
           'holdout_accessed':False},
 'runtime':{'python':platform.python_version(),'numpy':np.__version__,
            'pandas':pd.__version__,'sklearn':__import__('sklearn').__version__,
            'xgboost':__import__('xgboost').__version__,'shap':shap.__version__},
 'detector_thresholds_from_calibration':thresholds,
 'gate_thresholds_from_calibration':gate_params,
 'candidate_sampling_audit':candidate_selection,
 'detector_results':detector_scores,
 'dynamic_gate_results':dynamic_gate_results,
 'paired_clean_alert_cohort_diagnostic':paired_cohort_diagnostic,
 'limitations':['q_e measures completeness only, not source trust/context',
    'm_d is a margin, not empirically established detection reliability',
    'r_x is local SHAP stability under a small numeric probe',
    'synthetic missingness is not a measured real-world failure rate',
    'v2 corrects evaluation but retains v1 thresholds; effectiveness is not presumed',
    'M5/M6 and host-independent validation not tested',
    'file2 holdout not accessed'],
 'next_gate':'Examine candidate tradeoff and evidential validity before any heldout evaluation.'}
if OUTFILE.exists():raise FileExistsError('Keluaran v2 sudah ada; STOP sebelum menimpa.')
OUT.mkdir(parents=True,exist_ok=True)
OUTFILE.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print('Simpan hasil v2:',OUTFILE)
print('HOLDOUT BELUM DIGUNAKAN.')
